# Formula 1 — Comprehensive Statistical Portfolio (1950–2025)

> This report covers 10 analytical domains, 40+ interactive visualizations, and over 100 distinct metrics. Sections are self-contained and ordered from broad records to fine-grained performance indicators.

**Sections:**
1. Setup & Data Overview
2. Driver Records — Volume
3. Driver Efficiency — Era-Adjusted Ratios
4. Driver Racecraft — Positions Gained & Consistency
5. Prestige Achievements — Hat Tricks, Grand Slams, Pole Conversion
6. Constructor Dominance & Reliability
7. Circuit DNA — Attrition, Strategy & Pole Importance
8. Historical Trends — Reliability, Parity & Expansion
9. Nationality & Geography
10. Curiosities & Edge Cases


## 0. Setup & Data Overview

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.config import PROCESSED_DATA_DIR

# ---- Visual constants ----
TEMPLATE = "plotly_dark"
C_RED    = "#E10600"
C_GOLD   = "#FFD700"
C_SILVER = "#C0C0C0"
C_BLUE   = "#4FC3F7"

px.defaults.template = TEMPLATE

# ---- Load data ----
df = pd.read_parquet(PROCESSED_DATA_DIR / "results.parquet")
df_drivers = pd.read_parquet(PROCESSED_DATA_DIR / "drivers.parquet")

# ---- Key derived columns ----
# Did Not Finish detection: status not in completion signals
FINISH_STATUSES = {'Finished', '+1 Lap', '+2 Laps', '+3 Laps', '+4 Laps',
                   '+5 Laps', '+6 Laps', '+7 Laps', '+8 Laps', '+9 Laps', 'Lapped'}
df['finished']        = df['status'].isin(FINISH_STATUSES)
df['is_dnf']          = ~df['finished'] & df['positionText'].isin(['R', 'D', 'E', 'W', 'F', 'N'])
df['on_podium']       = df['position'] <= 3
df['is_win']          = df['position'] == 1
df['is_pole']         = df['grid'] == 1
df['in_points']       = df['position'] <= 10   # modern top-10 scoring
df['fastest_lap']     = pd.to_numeric(df['fastest_lap_rank'], errors='coerce') == 1
df['decade']          = (df['season'] // 10) * 10
df['positions_gained']= df['grid'] - df['position']   # positive = gained

# Merge date-of-birth for age calculations
df = df.merge(df_drivers[['driverId', 'dateOfBirth']], on='driverId', how='left')
df['date'] = pd.to_datetime(df['date'])
df['dateOfBirth'] = pd.to_datetime(df['dateOfBirth'])
df['driver_age_at_race'] = (df['date'] - df['dateOfBirth']).dt.days / 365.25

print(f"Records loaded  : {len(df):,}")
print(f"Seasons covered : {df['season'].min()} – {df['season'].max()}")
print(f"Unique drivers  : {df['driver_fullname'].nunique()}")
print(f"Unique events   : {df['raceName'].nunique()}")
print(f"Unique teams    : {df['constructor_name'].nunique()}")


## 1. Driver Records — Volume

*Raw career totals: the absolute figures every fan knows.*

In [ ]:

# ---- Aggregate per driver ----
drv = df.groupby('driver_fullname').agg(
    entries      = ('position', 'count'),
    wins         = ('is_win', 'sum'),
    podiums      = ('on_podium', 'sum'),
    poles        = ('is_pole', 'sum'),
    fastest_laps = ('fastest_lap', 'sum'),
    total_points = ('points', 'sum'),
    seasons      = ('season', 'nunique'),
    dnfs         = ('is_dnf', 'sum'),
    top10s       = ('in_points', 'sum'),
).reset_index().rename(columns={'driver_fullname': 'driver'})

drv = drv.sort_values('wins', ascending=False)
TOP_N = 25

# Chart 1a — Career Wins (Top 25)
fig = px.bar(
    drv.head(TOP_N).sort_values('wins'),
    x='wins', y='driver', orientation='h',
    color='wins', color_continuous_scale='Reds',
    title=f'Career Grand Prix Victories — Top {TOP_N}',
    labels={'wins': 'Victories', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 1b — Career Podiums
fig = px.bar(
    drv.sort_values('podiums', ascending=False).head(TOP_N).sort_values('podiums'),
    x='podiums', y='driver', orientation='h',
    color='podiums', color_continuous_scale='RdYlGn',
    title=f'Career Podium Finishes — Top {TOP_N}',
    labels={'podiums': 'Podiums', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 1c — Career Points
fig = px.bar(
    drv.sort_values('total_points', ascending=False).head(TOP_N).sort_values('total_points'),
    x='total_points', y='driver', orientation='h',
    color='total_points', color_continuous_scale='Blues',
    title=f'Career Championship Points — Top {TOP_N}',
    labels={'total_points': 'Points', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 1d — Most GP Entries (Longevity)
fig = px.bar(
    drv.sort_values('entries', ascending=False).head(TOP_N).sort_values('entries'),
    x='entries', y='driver', orientation='h',
    color='entries', color_continuous_scale='Purples',
    title=f'Career Grand Prix Entries — Top {TOP_N}',
    labels={'entries': 'GP Starts', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Table — Combined Top 15
drv.head(15)[['driver','entries','wins','podiums','poles','fastest_laps','total_points','seasons']]


## 2. Driver Efficiency — Era-Adjusted Ratios

*Raw totals favour modern era drivers with more races. Ratios level the playing field.*

*Minimum 20 GP starts applied to remove statistical outliers.*

In [ ]:

# Efficiency metrics (min 20 entries)
eff = drv[drv['entries'] >= 20].copy()
eff['win_pct']     = (eff['wins']    / eff['entries'] * 100).round(2)
eff['podium_pct']  = (eff['podiums'] / eff['entries'] * 100).round(2)
eff['pole_pct']    = (eff['poles']   / eff['entries'] * 100).round(2)
eff['pts_per_gp']  = (eff['total_points'] / eff['entries']).round(2)
eff['dnf_pct']     = (eff['dnfs']    / eff['entries'] * 100).round(2)
eff['top10_pct']   = (eff['top10s']  / eff['entries'] * 100).round(2)

# Chart 2a — Win Percentage (min 20 entries)
top_win_pct = eff.sort_values('win_pct', ascending=False).head(20)
fig = px.bar(
    top_win_pct.sort_values('win_pct'),
    x='win_pct', y='driver', orientation='h',
    color='win_pct', color_continuous_scale='Reds',
    title='Win Percentage per Career Start (Min. 20 Entries)',
    labels={'win_pct': 'Win %', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 2b — Scatter: Win% vs Entries (bubble = total wins)
fig = px.scatter(
    eff.sort_values('wins', ascending=False).head(50),
    x='entries', y='win_pct',
    size='wins', size_max=50,
    hover_name='driver',
    color='win_pct', color_continuous_scale='Reds',
    title='Win Efficiency vs Career Longevity',
    labels={'entries': 'GP Entries', 'win_pct': 'Win %'}
)
fig.update_coloraxes(showscale=False)
fig.show()

# Chart 2c — Podium percentage
top_pod_pct = eff.sort_values('podium_pct', ascending=False).head(20)
fig = px.bar(
    top_pod_pct.sort_values('podium_pct'),
    x='podium_pct', y='driver', orientation='h',
    color='podium_pct', color_continuous_scale='RdYlGn',
    title='Podium Percentage per Career Start (Min. 20 Entries)',
    labels={'podium_pct': 'Podium %', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 2d — Points per GP start
top_pts = eff.sort_values('pts_per_gp', ascending=False).head(20)
fig = px.bar(
    top_pts.sort_values('pts_per_gp'),
    x='pts_per_gp', y='driver', orientation='h',
    color='pts_per_gp', color_continuous_scale='Blues',
    title='Average Points per GP Start (Min. 20 Entries)',
    labels={'pts_per_gp': 'Pts/GP', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

eff.sort_values('win_pct', ascending=False).head(15)[
    ['driver','entries','win_pct','podium_pct','pole_pct','pts_per_gp','dnf_pct']
]


## 3. Driver Racecraft — Positions Gained & Consistency

*Who makes up ground on race day? Who protects their grid slot?*

In [ ]:

# Only races where grid is valid (not 0 = pit lane start)
df_race = df[(df['grid'] > 0) & (df['position'] > 0)].copy()

racecraft = df_race.groupby('driver_fullname').agg(
    entries         = ('positions_gained', 'count'),
    avg_gained      = ('positions_gained', 'mean'),
    total_gained    = ('positions_gained', 'sum'),
    avg_grid        = ('grid', 'mean'),
    avg_finish      = ('position', 'mean'),
).reset_index().rename(columns={'driver_fullname': 'driver'})

racecraft['avg_gained'] = racecraft['avg_gained'].round(2)
racecraft['avg_grid']   = racecraft['avg_grid'].round(2)
racecraft['avg_finish'] = racecraft['avg_finish'].round(2)

# Min 50 races for reliability
rc_filtered = racecraft[racecraft['entries'] >= 50]

# Chart 3a — Top climbers (best avg positions gained)
climbers = rc_filtered.sort_values('avg_gained', ascending=False).head(20)
fig = px.bar(
    climbers.sort_values('avg_gained'),
    x='avg_gained', y='driver', orientation='h',
    color='avg_gained', color_continuous_scale='Greens',
    title='Average Positions Gained per Race (Min. 50 Starts)',
    labels={'avg_gained': 'Avg Pos. Gained', 'driver': ''}
)
fig.add_vline(x=0, line_dash='dash', line_color='white', opacity=0.4)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 3b — Avg grid vs avg finish (dot below diagonal = improves)
top_rc = rc_filtered.sort_values('entries', ascending=False).head(40)
fig = px.scatter(
    top_rc, x='avg_grid', y='avg_finish',
    hover_name='driver', size='entries',
    color='avg_gained', color_continuous_scale='RdYlGn',
    title='Average Grid vs Average Finish Position (Top 40 by GP Entries)',
    labels={'avg_grid': 'Avg Grid Position', 'avg_finish': 'Avg Finish Position'}
)
fig.add_shape(type='line', x0=1, y0=1, x1=20, y1=20,
              line=dict(color='white', dash='dash', width=1))
fig.add_annotation(x=15, y=14, text='Grid = Finish', showarrow=False,
                   font=dict(color='white', size=10))
fig.update_coloraxes(colorbar_title='Avg Gained')
fig.show()

rc_filtered.sort_values('avg_gained', ascending=False).head(15)[
    ['driver','entries','avg_grid','avg_finish','avg_gained','total_gained']
]


## 4. Prestige Achievements

- **Hat Trick**: Pole position + Fastest Lap + Win in the same race
- **Grand Slam**: Pole + Win + Fastest Lap + Led every lap
  *(Approximated here as Pole + Win + Fastest Lap, since lap-level leadership is not in this dataset)*
- **Pole Conversion Rate**: % of poles that resulted in a race win

In [ ]:

# Hat tricks = pole + win + fastest lap in SAME race
hat_tricks = df[(df['is_pole']) & (df['is_win']) & (df['fastest_lap'])].groupby(
    'driver_fullname').size().reset_index(name='hat_tricks')

# Pole Conversion Rate
poles_df   = df[df['is_pole']].groupby('driver_fullname').size().reset_index(name='pole_starts')
pole_wins  = df[(df['is_pole']) & (df['is_win'])].groupby('driver_fullname').size().reset_index(name='pole_wins')
pole_conv  = poles_df.merge(pole_wins, on='driver_fullname', how='left').fillna(0)
pole_conv['pole_conv_pct'] = (pole_conv['pole_wins'] / pole_conv['pole_starts'] * 100).round(2)
pole_conv = pole_conv[pole_conv['pole_starts'] >= 3].sort_values('pole_conv_pct', ascending=False)

# Chart 4a — Hat Tricks
fig = px.bar(
    hat_tricks.sort_values('hat_tricks', ascending=False).head(15).sort_values('hat_tricks'),
    x='hat_tricks', y='driver_fullname', orientation='h',
    color='hat_tricks', color_continuous_scale='Oranges',
    title='Hat Tricks: Pole + Fastest Lap + Win — Career Total',
    labels={'hat_tricks': 'Hat Tricks', 'driver_fullname': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Chart 4b — Pole Conversion Rate (min 3 poles)
fig = px.bar(
    pole_conv.head(20).sort_values('pole_conv_pct'),
    x='pole_conv_pct', y='driver_fullname', orientation='h',
    color='pole_conv_pct', color_continuous_scale='Reds',
    title='Pole Position Conversion Rate (Min. 3 Pole Positions)',
    labels={'pole_conv_pct': 'Pole to Win %', 'driver_fullname': ''},
    hover_data=['pole_starts', 'pole_wins']
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

print("=== Hat Trick Leaders ===")
print(hat_tricks.sort_values('hat_tricks', ascending=False).head(10).to_string(index=False))
print()
print("=== Pole Conversion Leaders ===")
print(pole_conv.head(15)[['driver_fullname','pole_starts','pole_wins','pole_conv_pct']].to_string(index=False))


## 5. Age Records

*Youngest and oldest to achieve key milestones.*

In [ ]:

wins_df    = df[df['is_win'] & df['driver_age_at_race'].notna()].copy()
podiums_df = df[df['on_podium'] & df['driver_age_at_race'].notna()].copy()
poles_df_a = df[df['is_pole'] & df['driver_age_at_race'].notna()].copy()

# Youngest/Oldest winners
youngest_wins = wins_df.loc[wins_df.groupby('driver_fullname')['driver_age_at_race'].idxmin()]
youngest_wins = youngest_wins.sort_values('driver_age_at_race').head(10)[
    ['driver_fullname','season','raceName','driver_age_at_race']].rename(
    columns={'driver_age_at_race': 'age_years'})
youngest_wins['age_years'] = youngest_wins['age_years'].round(2)

oldest_wins = wins_df.loc[wins_df.groupby('driver_fullname')['driver_age_at_race'].idxmax()]
oldest_wins = oldest_wins.sort_values('driver_age_at_race', ascending=False).head(10)[
    ['driver_fullname','season','raceName','driver_age_at_race']].rename(
    columns={'driver_age_at_race': 'age_years'})
oldest_wins['age_years'] = oldest_wins['age_years'].round(2)

# Chart 5a — Youngest winners
fig = go.Figure([go.Bar(
    x=youngest_wins['age_years'],
    y=youngest_wins['driver_fullname'] + ' (' + youngest_wins['season'].astype(str) + ')',
    orientation='h',
    marker_color=C_RED,
    text=youngest_wins['age_years'].astype(str) + ' yrs',
    textposition='outside'
)])
fig.update_layout(
    title='Youngest Race Winners in F1 History',
    template=TEMPLATE, xaxis_title='Age at First Win (years)',
    yaxis={'categoryorder': 'total ascending'}, height=500
)
fig.show()

# Chart 5b — Oldest winners
fig = go.Figure([go.Bar(
    x=oldest_wins['age_years'],
    y=oldest_wins['driver_fullname'] + ' (' + oldest_wins['season'].astype(str) + ')',
    orientation='h',
    marker_color=C_GOLD,
    text=oldest_wins['age_years'].astype(str) + ' yrs',
    textposition='outside'
)])
fig.update_layout(
    title='Oldest Race Winners in F1 History (Last Win)',
    template=TEMPLATE, xaxis_title='Age at Last Win (years)',
    yaxis={'categoryorder': 'total ascending'}, height=500
)
fig.show()

print("=== 10 Youngest Race Winners ===")
print(youngest_wins.to_string(index=False))
print()
print("=== 10 Oldest Race Winners (at Last Win) ===")
print(oldest_wins.to_string(index=False))


## 6. Constructor Dominance & Reliability

In [ ]:

const = df.groupby('constructor_name').agg(
    entries      = ('position', 'count'),
    wins         = ('is_win', 'sum'),
    podiums      = ('on_podium', 'sum'),
    poles        = ('is_pole', 'sum'),
    fastest_laps = ('fastest_lap', 'sum'),
    total_points = ('total_points', 'sum') if 'total_points' in df.columns else ('points', 'sum'),
    dnfs         = ('is_dnf', 'sum'),
    seasons      = ('season', 'nunique'),
    drivers_used = ('driverId', 'nunique'),
).reset_index().rename(columns={'constructor_name': 'constructor'})

const = const.sort_values('wins', ascending=False)

# Efficiency
const['win_pct']    = (const['wins']   / const['entries'] * 100).round(2)
const['dnf_pct']    = (const['dnfs']   / const['entries'] * 100).round(2)
const['pts_per_gp'] = (const['total_points'] / const['entries']).round(2)

# Chart 6a — Constructor Wins
fig = px.bar(
    const.head(15).sort_values('wins'),
    x='wins', y='constructor', orientation='h',
    color='wins', color_continuous_scale='Reds',
    title='All-Time Constructor Grand Prix Victories — Top 15',
    labels={'wins': 'Victories', 'constructor': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 6b — Constructor Reliability (DNF %)
reliable = const[const['entries'] >= 50].sort_values('dnf_pct')
fig = px.bar(
    reliable.head(20).sort_values('dnf_pct', ascending=False),
    x='dnf_pct', y='constructor', orientation='h',
    color='dnf_pct', color_continuous_scale='RdYlGn_r',
    title='Constructor Reliability: DNF Rate (Min. 50 Entries)',
    labels={'dnf_pct': 'DNF %', 'constructor': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# Chart 6c — Constructor Win % (min 50 entries)
fig = px.bar(
    const[const['entries'] >= 50].sort_values('win_pct', ascending=False).head(15).sort_values('win_pct'),
    x='win_pct', y='constructor', orientation='h',
    color='win_pct', color_continuous_scale='Reds',
    title='Constructor Win Percentage (Min. 50 Entries)',
    labels={'win_pct': 'Win %', 'constructor': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()

# Chart 6d — Constructor points by season (Top 8 all-time teams)
top8_const = const.head(8)['constructor'].tolist()
season_pts = df[df['constructor_name'].isin(top8_const)].groupby(
    ['season', 'constructor_name'])['points'].sum().reset_index()
fig = px.line(season_pts, x='season', y='points', color='constructor_name',
              title='Constructor Points per Season — Top 8 Teams',
              labels={'points': 'Season Points', 'season': 'Year', 'constructor_name': 'Constructor'})
fig.update_layout(height=500)
fig.show()

const.head(15)[['constructor','entries','wins','win_pct','podiums','poles','dnf_pct','seasons','drivers_used']]


## 7. Circuit DNA — Attrition, Strategy & Pole Importance

In [ ]:

# Circuit-level attrition
circ = df.groupby(['circuitId', 'raceName']).agg(
    total_starters = ('position', 'count'),
    dnfs           = ('is_dnf', 'sum'),
    pole_wins      = ('is_pole', lambda x: (x & df.loc[x.index, 'is_win']).sum()),
    races_held     = ('round', 'nunique'),
).reset_index()
circ['attrition_pct'] = (circ['dnfs'] / circ['total_starters'] * 100).round(2)
circ['pole_win_pct']  = (circ['pole_wins'] / circ['races_held'] * 100).round(2)

# Chart 7a — Most punishing circuits
fig = px.bar(
    circ[circ['races_held'] >= 5].sort_values('attrition_pct', ascending=False).head(20),
    x='attrition_pct', y='raceName', orientation='h',
    color='attrition_pct', color_continuous_scale='Reds',
    title='Circuit Severity: Average DNF Rate (Min. 5 Editions)',
    labels={'attrition_pct': 'DNF %', 'raceName': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Chart 7b — Pole-to-Win conversion by circuit (Monaco factor)
fig = px.bar(
    circ[circ['races_held'] >= 10].sort_values('pole_win_pct', ascending=False).head(20).sort_values('pole_win_pct'),
    x='pole_win_pct', y='raceName', orientation='h',
    color='pole_win_pct', color_continuous_scale='Oranges',
    title='Pole Position Win Conversion by Circuit (Min. 10 Editions)',
    labels={'pole_win_pct': 'Pole to Win %', 'raceName': ''},
    hover_data=['races_held']
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=700)
fig.show()

# Most dominant driver per circuit
circuit_wins = df[df['is_win']].groupby(['raceName', 'driver_fullname']).size().reset_index(name='wins')
most_dominant = circuit_wins.loc[circuit_wins.groupby('raceName')['wins'].idxmax()].rename(
    columns={'driver_fullname': 'most_wins_driver'})
most_dominant = most_dominant[most_dominant['wins'] >= 2].sort_values('wins', ascending=False)

print("=== Most Dominant Drivers by Circuit ===")
print(most_dominant.head(20).to_string(index=False))


## 8. Historical Trends — Reliability, Parity & Calendar Expansion

In [ ]:

# Chart 8a — DNF rate by decade
dnf_decade = df.groupby('decade').agg(
    total   = ('is_dnf', 'count'),
    dnfs    = ('is_dnf', 'sum')
).reset_index()
dnf_decade['dnf_pct'] = (dnf_decade['dnfs'] / dnf_decade['total'] * 100).round(2)

fig = px.bar(dnf_decade, x='decade', y='dnf_pct',
             color='dnf_pct', color_continuous_scale='RdYlGn_r',
             title='DNF Rate Evolution by Decade (Technical Reliability)',
             labels={'dnf_pct': 'DNF %', 'decade': 'Decade'})
fig.update_coloraxes(showscale=False)
fig.show()

# Chart 8b — Races per season
gps_per_season = df.groupby('season')['round'].max().reset_index(name='num_gps')
fig = px.area(gps_per_season, x='season', y='num_gps',
              title='Formula 1 Calendar Expansion: Grand Prix per Season',
              labels={'num_gps': 'Number of Races', 'season': 'Year'})
fig.update_traces(line_color=C_RED, fillcolor='rgba(225,6,0,0.2)')
fig.show()

# Chart 8c — Unique winners per season (competitiveness index)
unique_winners = df[df['is_win']].groupby('season')['driver_fullname'].nunique().reset_index(name='unique_winners')
fig = px.line(unique_winners, x='season', y='unique_winners', markers=True,
              title='Competitive Diversity: Unique Race Winners per Season',
              labels={'unique_winners': 'Distinct Winners', 'season': 'Year'})
fig.add_hline(y=unique_winners['unique_winners'].mean(), line_dash='dash',
              annotation_text=f"Average: {unique_winners['unique_winners'].mean():.1f}",
              annotation_position='top right')
fig.update_traces(line_color=C_GOLD)
fig.show()

# Chart 8d — Most dominant season (highest win % by a single driver)
season_dom = df.groupby(['season', 'driver_fullname']).agg(
    wins    = ('is_win', 'sum'),
    entries = ('position', 'count')
).reset_index()
season_dom['win_pct'] = season_dom['wins'] / season_dom['entries'] * 100
# Find the most dominant driver each season
best_season = season_dom.loc[season_dom.groupby('season')['win_pct'].idxmax()].sort_values('win_pct', ascending=False)

print("=== Most Dominant Single-Season Performances (Win %) ===")
print(best_season.head(15)[['season','driver_fullname','wins','entries','win_pct']].to_string(index=False))


## 9. Nationality & Geography

In [ ]:

# Nationality requires driver table nationality column
if 'nationality' in df_drivers.columns:
    nat_df = df.merge(df_drivers[['driverId','nationality']], on='driverId', how='left')
    nat_wins = nat_df[nat_df['is_win']].groupby('nationality').size().reset_index(name='wins')
    nat_entries = nat_df.groupby('nationality')['position'].count().reset_index(name='entries')
    nat_stats = nat_wins.merge(nat_entries, on='nationality')
    nat_stats['win_pct'] = (nat_stats['wins'] / nat_stats['entries'] * 100).round(2)
    nat_stats = nat_stats.sort_values('wins', ascending=False)

    # Chart 9a — Wins by nationality
    fig = px.bar(
        nat_stats.head(15).sort_values('wins'),
        x='wins', y='nationality', orientation='h',
        color='wins', color_continuous_scale='Blues',
        title='Grand Prix Victories by Driver Nationality — Top 15',
        labels={'wins': 'Victories', 'nationality': ''}
    )
    fig.update_coloraxes(showscale=False)
    fig.update_layout(height=550)
    fig.show()

    # Chart 9b — Pie chart of wins distribution
    fig = px.pie(nat_stats.head(10), values='wins', names='nationality',
                 title='Share of Grand Prix Victories by Nation (Top 10)',
                 color_discrete_sequence=px.colors.qualitative.Dark24)
    fig.show()
else:
    print("Nationality data not available in drivers table.")


## 10. Curiosities & Edge Cases

*Rare feats, unlucky records, and unusual statistical patterns.*

In [ ]:

# --- 10a: Most podiums without a win ---
no_wins = drv[drv['wins'] == 0].sort_values('podiums', ascending=False).head(10)
print("=== Most Podiums Without a Single Victory ===")
print(no_wins[['driver','entries','podiums','total_points']].to_string(index=False))

# Chart 10a
fig = px.bar(
    no_wins.sort_values('podiums'),
    x='podiums', y='driver', orientation='h',
    color='podiums', color_continuous_scale='Oranges',
    title='Most Podiums Without a Race Win',
    labels={'podiums': 'Career Podiums', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.show()

# --- 10b: Most constructors driven for (most nomadic) ---
constructors_driven = df.groupby('driver_fullname')['constructor_name'].nunique().reset_index(
    name='num_constructors').sort_values('num_constructors', ascending=False)
print()
print("=== Most Constructors Driven For ===")
print(constructors_driven.head(10).to_string(index=False))

# --- 10c: Most GP starts before first win (perseverance) ---
first_win_round = df[df['is_win']].groupby('driver_fullname').agg(
    first_win_season = ('season', 'min')
).reset_index()
# Count entries before first win
races_before_win = []
for _, row in first_win_round.iterrows():
    driver = row['driver_fullname']
    first_win_yr = row['first_win_season']
    entries_before = df[(df['driver_fullname'] == driver) & (df['season'] <= first_win_yr)].shape[0]
    races_before_win.append({'driver': driver, 'first_win_year': first_win_yr, 'starts_to_first_win': entries_before})

rbw_df = pd.DataFrame(races_before_win).sort_values('starts_to_first_win', ascending=False).head(15)

print()
print("=== Most GP Starts Before First Victory ===")
print(rbw_df.to_string(index=False))

# Chart 10c
fig = px.bar(
    rbw_df.sort_values('starts_to_first_win'),
    x='starts_to_first_win', y='driver', orientation='h',
    color='starts_to_first_win', color_continuous_scale='Purples',
    title='GP Starts Before First Career Victory',
    labels={'starts_to_first_win': 'Starts Before First Win', 'driver': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=550)
fig.show()

# --- 10d: Retirement cause breakdown ---
dnf_causes = df[df['is_dnf']].groupby('status').size().sort_values(ascending=False).head(20)
fig = px.bar(
    dnf_causes.reset_index().rename(columns={'index': 'cause', 'status': 'cause', 0: 'count'}),
    x=dnf_causes.values, y=dnf_causes.index, orientation='h',
    color=dnf_causes.values, color_continuous_scale='Reds',
    title='DNF Causes — Historical Breakdown',
    labels={'x': 'Occurrences', 'y': 'Retirement Cause'}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=600)
fig.show()

# --- 10e: Comeback wins (largest grid position that resulted in a win) ---
comeback_wins = df[df['is_win']].sort_values('grid', ascending=False).head(15)[
    ['driver_fullname', 'season', 'raceName', 'grid', 'position']
].rename(columns={'driver_fullname': 'driver', 'grid': 'start_position'})

print()
print("=== Greatest Comeback Victories (Started Furthest From Pole) ===")
print(comeback_wins.to_string(index=False))
fig = px.bar(
    comeback_wins.sort_values('start_position'),
    x='start_position',
    y=comeback_wins['driver'] + ' (' + comeback_wins['season'].astype(str) + ' ' + comeback_wins['raceName'] + ')',
    orientation='h',
    color='start_position', color_continuous_scale='Reds',
    title='Greatest Comeback Victories (Starting Position When Winning)',
    labels={'x': 'Grid Position', 'y': ''}
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=500)
fig.show()
